# Intro: Local RAG Evaluation

This notebook demonstrates a lightweight RAG evaluation loop with deterministic groundedness and answer-correctness scores. It also compares a stricter config to a more lenient one without calling any external services.

In [ ]:
from pathlib import Path
import json

from eval_harness.dataset import load_jsonl_cases
from eval_harness.judges import MockJudge, load_rubric

root = Path.cwd()
cases = load_jsonl_cases(root / "tests" / "evals" / "datasets" / "rag_cases.jsonl")
outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_model_outputs.json").read_text())
fixture_outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_judge_outputs.json").read_text())
rubric = load_rubric(root / "tests" / "evals" / "rubrics" / "answer_quality.yaml")
judge = MockJudge(fixture_outputs)

rows = []
for case in cases:
    judged = judge.judge(case, outputs[case.case_id], rubric)
    rows.append(
        {
            "case_id": case.case_id,
            "groundedness": judged.scores["groundedness"],
            "answer_correctness": judged.scores["answer_correctness"],
            "passed": judged.passed,
        }
    )

strict_config = {"min_groundedness": 1.0, "min_answer_correctness": 1.0}
lenient_config = {"min_groundedness": 0.5, "min_answer_correctness": 0.5}

comparison = {
    "rows": rows,
    "strict_passes": [
        row["case_id"]
        for row in rows
        if row["groundedness"] >= strict_config["min_groundedness"]
        and row["answer_correctness"] >= strict_config["min_answer_correctness"]
    ],
    "lenient_passes": [
        row["case_id"]
        for row in rows
        if row["groundedness"] >= lenient_config["min_groundedness"]
        and row["answer_correctness"] >= lenient_config["min_answer_correctness"]
    ],
}

comparison
